# EDA SINAN — Brasil

Dados de anos encerrados, agregados por residência. Execute primeiro `python -m utils.deng_download` e `python -m utils.analysis_pipeline`. As saídas anteriores foram invalidadas após correção da classificação histórica. `DENGUE` não é diagnóstico final: usamos `CON_CLASSI` (até 2006) e `CLASSI_FIN` (demais anos).

**Período anual:** `ano` nos contratos SINAN identifica o ano-fonte (`periodo_referencia = ano_fonte_sinan`), não necessariamente um ano civil completo. Datas em anos adjacentes são preservadas e auditadas. A população usa o ano de referência; as séries diárias climáticas permanecem separadas, pela data efetiva de notificação.

A coorte original inclui uma notificação de 2008 com data corrompida. A execução usa `allow_undated=True`: o registro permanece no período-fonte anual e na auditoria, sem inventar dia ou mês; gráficos por data não o incluem.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "utils/analysis_pipeline.py").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import polars as pl
from IPython.display import display

import plotly.express as px
from utils.analysis_pipeline import PROCESSED_DIR, build_sinan_outputs, scan_sinan
outputs = build_sinan_outputs(allow_undated=True)
panel = pl.read_parquet(PROCESSED_DIR / "sinan_populacao_municipio_ano.parquet")
raw = scan_sinan()

/Users/thitav/Documents/Pysus-IBGE/utils/analysis_pipeline.py:410: UserWarning: 1 registro(s) sem data válida mantido(s) apenas pelo ano-fonte; a exportação mensal conserva ano/mês civil desconhecido. Consulte auditoria_sinan_datas_invalidas.csv.
  warnings.warn(


## Qualidade e universo

Notificações não são necessariamente casos confirmados. Sem fonte anual não se infere zero. Municípios sem notificação recebem zero somente em anos com fonte presente. Registros sem residência válida ficam na auditoria; não são distribuídos entre municípios.

In [2]:
display(outputs["qualidade"])
annual = panel.group_by("ano").agg(pl.col("notificacoes").sum(), pl.col("casos_confirmados").sum()).sort("ano")
display(annual)
px.line(annual.to_pandas(), x="ano", y=["notificacoes", "casos_confirmados"], markers=True).show()

arquivo,ano_arquivo,registros,codigo_municipio_residencia_invalido,codigo_municipio_notificacao_invalido,codigo_municipio_invalido,municipio_residencia_notificacao_divergente,data_invalida,ano_informado_divergente,ano_data_divergente
str,i32,u32,u32,u32,u32,u32,u32,u32,u32
"""DENGBR00.parquet""",2000,172855,1,0,1,7207,0,48,0
"""DENGBR01.parquet""",2001,488590,34,0,34,14012,0,243,0
"""DENGBR02.parquet""",2002,897093,255,0,255,42574,0,202,0
"""DENGBR03.parquet""",2003,416609,311,0,311,16742,0,20,0
"""DENGBR04.parquet""",2004,136867,67,0,67,5507,0,0,0
…,…,…,…,…,…,…,…,…,…
"""DENGBR19.parquet""",2019,2261956,52,0,52,138127,0,1809,1809
"""DENGBR20.parquet""",2020,1495117,35,0,35,69046,0,6800,6800
"""DENGBR21.parquet""",2021,1024635,77,0,77,55955,0,1466,1466


ano,notificacoes,casos_confirmados
i32,u64,u64
2000,172854,99349
2001,488556,286610
2002,896838,481260
2003,416298,227424
2004,136800,63483
…,…,…
2019,2261904,1311737
2020,1495082,793006
2021,1024558,455711


## Sazonalidade

Grade completa de dias dos anos-fonte, incluindo dias sem notificação registrada. A média mensal abaixo é por dia de calendário; a soma mensal é publicada separadamente.

O gráfico diário conserva datas adjacentes aos anos-fonte. A comparação mensal se limita aos anos-fonte presentes; anos adjacentes apenas parcialmente observados não entram nessa comparação. Datas ilegíveis não entram em nenhum gráfico de calendário.

In [3]:
undated_count = raw.filter(pl.col("data_notificacao").is_null() & pl.col("codigo_municipio").is_not_null()).select(pl.len()).collect().item()
print("Datas desconhecidas: mantidas no ano-fonte e excluídas dos gráficos de calendário:", undated_count)
from datetime import date
years = outputs["qualidade"]["ano_arquivo"].to_list()
calendar = pl.concat([pl.DataFrame({"Dia": pl.date_range(date(y, 1, 1), date(y, 12, 31), eager=True)}) for y in years])
daily = raw.filter(pl.col("codigo_municipio").is_not_null()).group_by("data_notificacao").agg(pl.len().alias("notificacoes")).rename({"data_notificacao":"Dia"}).collect()
daily = calendar.join(daily.filter(pl.col("Dia").is_not_null()), on="Dia", how="full", coalesce=True, validate="1:1").with_columns(pl.col("notificacoes").fill_null(0)).sort("Dia")
monthly = daily.with_columns(pl.col("Dia").dt.year().alias("ano"), pl.col("Dia").dt.month().alias("mes")).group_by("ano", "mes").agg(pl.col("notificacoes").sum().alias("notificacoes_mes"), pl.col("notificacoes").mean().alias("media_por_dia_calendario")).sort("ano", "mes")
monthly = monthly.filter(pl.col("ano").is_in(years))
display(monthly)

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
df_notificacoes = daily.rename({"notificacoes": "Notificações"}).to_pandas()
df_notificacoes["Dia"] = pd.to_datetime(df_notificacoes["Dia"])
df_sazonalidade = monthly.rename({"ano": "Ano", "mes": "Mês", "media_por_dia_calendario": "Notificações"}).to_pandas()

Datas desconhecidas: mantidas no ano-fonte e excluídas dos gráficos de calendário: 1


ano,mes,notificacoes_mes,media_por_dia_calendario
i32,i8,u32,f64
2000,1,14207,458.290323
2000,2,24740,853.103448
2000,3,32163,1037.516129
2000,4,28897,963.233333
2000,5,24669,795.774194
…,…,…,…
2023,8,33020,1065.16129
2023,9,24391,813.033333
2023,10,28138,907.677419


In [4]:
media = df_notificacoes['Notificações'].mean()

fig = px.line(
    df_notificacoes,
    x='Dia',
    y='Notificações'
)

fig.add_hline(
    y=media,
    line_dash='dash',
    annotation_text=f"Média: {media:.0f}",
    annotation_position="top left"
)

fig.update_traces(line=dict(width=1))

fig.show()

In [5]:
fig = px.line(
    df_sazonalidade,
    x="Mês",
    y="Notificações",
    color="Ano",
    markers=True,
    labels={
        "Mês": "Mês",
        "Notificações": "Notificações",
        "Ano": "Ano"
    },
    title="Sazonalidade por ano — média por dia de calendário"
)

fig.update_layout(
    xaxis=dict(
        tickmode="array",
        tickvals=list(range(1, 13)),
        ticktext=[
            "Jan", "Fev", "Mar", "Abr",
            "Mai", "Jun", "Jul", "Ago",
            "Set", "Out", "Nov", "Dez"
        ]
    )
)

fig.show()

In [6]:
from plotly.subplots import make_subplots

# Criar ano e mês
df_notificacoes["Ano"] = df_notificacoes["Dia"].dt.year
df_notificacoes["Mês"] = df_notificacoes["Dia"].dt.month

# Soma das notificações de cada mês de cada ano
df_mensal = (
    df_notificacoes
    .groupby(["Ano", "Mês"])["Notificações"]
    .sum()
    .reset_index()
)
df_mensal = df_mensal[df_mensal["Ano"].isin(years)]

nomes_meses = [
    "Janeiro", "Fevereiro", "Março", "Abril",
    "Maio", "Junho", "Julho", "Agosto",
    "Setembro", "Outubro", "Novembro", "Dezembro"
]

# Criar 12 gráficos
fig = make_subplots(
    rows=4,
    cols=3,
    subplot_titles=nomes_meses
)

for mes in range(1, 13):

    dados = df_mensal[df_mensal["Mês"] == mes]

    # Média histórica daquele mês
    media = dados["Notificações"].mean()

    linha = go.Scatter(
        x=dados["Ano"],
        y=dados["Notificações"],
        mode="lines+markers",
        name="Notificações",
        showlegend=False
    )

    media_linha = go.Scatter(
        x=dados["Ano"],
        y=[media] * len(dados),
        mode="lines",
        name="Média",
        line=dict(dash="dash"),
        showlegend=False
    )

    linha_idx = (mes - 1) // 3 + 1
    coluna_idx = (mes - 1) % 3 + 1

    fig.add_trace(
        linha,
        row=linha_idx,
        col=coluna_idx
    )

    fig.add_trace(
        media_linha,
        row=linha_idx,
        col=coluna_idx
    )

fig.update_layout(
    title="Tendência das notificações por mês ao longo dos anos",
    height=900,
    width=1200
)

fig.show()

## Classificação, raça/cor, sexo e evolução das notificações

In [7]:
summary = raw.group_by("status_classificacao").len(name="Contagem").sort("Contagem", descending=True).collect().to_pandas()
summary["status_classificacao"] = summary["status_classificacao"].fillna("Não informado")
fig = px.bar(summary, x="status_classificacao", y="Contagem", text="Contagem", color="status_classificacao",
             title="Notificações por classificação final — anos-fonte disponíveis",
             labels={"Contagem": "Número de notificações", "status_classificacao": "Classificação final"})
fig.update_traces(textposition="outside")
fig.show()

In [8]:
summary = raw.group_by("sexo").len(name="Contagem").sort("Contagem", descending=True).collect().to_pandas()
summary["sexo"] = summary["sexo"].fillna("Não informado")
fig = px.bar(summary, x="sexo", y="Contagem", text="Contagem", color="sexo",
             title="Notificações por sexo — anos-fonte disponíveis",
             labels={"Contagem": "Número de notificações", "sexo": "Sexo"})
fig.update_traces(textposition="outside")
fig.show()

In [9]:
summary = raw.group_by("raca").len(name="Contagem").sort("Contagem", descending=True).collect().to_pandas()
summary["raca"] = summary["raca"].fillna("Não informado")
fig = px.bar(summary, x="raca", y="Contagem", text="Contagem", color="raca",
             title="Notificações por raça/cor — anos-fonte disponíveis",
             labels={"Contagem": "Número de notificações", "raca": "Raça/cor"})
fig.update_traces(textposition="outside")
fig.show()

In [10]:
summary = raw.group_by("evolucao").len(name="Contagem").sort("Contagem", descending=True).collect().to_pandas()
summary["evolucao"] = summary["evolucao"].fillna("Não informado")
fig = px.bar(summary, x="evolucao", y="Contagem", text="Contagem", color="evolucao",
             title="Notificações por evolução — anos-fonte disponíveis",
             labels={"Contagem": "Número de notificações", "evolucao": "Evolução"})
fig.update_traces(textposition="outside")
fig.show()

## Taxas municipais anuais

População residente do mesmo ano e chave IBGE. Nunca se divide a soma de vários anos por uma única população censitária. A seleção abaixo usa 2010; altere apenas para um ano com denominador disponível.

In [11]:
YEAR = 2010
municipal = panel.filter((pl.col("ano") == YEAR) & pl.col("correspondencia_populacao"))
if not municipal.height:
    raise ValueError(f"Sem denominador válido para {YEAR}")
display(municipal.select("codigo_municipio", "municipio", "populacao", "notificacoes", "casos_confirmados", "taxa_confirmados_100k").sort("taxa_confirmados_100k", descending=True))

states = municipal.with_columns(pl.col("codigo_municipio").str.slice(0, 2).alias("UF_codigo")).group_by("UF_codigo").agg(pl.col("notificacoes").sum(), pl.col("casos_confirmados").sum(), pl.col("populacao").sum()).with_columns((pl.col("casos_confirmados") / pl.col("populacao") * 100_000).alias("taxa_confirmados_100k"))
display(states)

codigo_municipio,municipio,populacao,notificacoes,casos_confirmados,taxa_confirmados_100k
str,str,u64,u64,u64,f64
"""120040""","""Rio Branco - AC""",336038,37100,32336,9622.72124
"""314050""","""Martinho Campos - MG""",12611,1252,1152,9134.882246
"""510719""","""Ribeirãozinho - MT""",2199,265,200,9095.043201
"""313880""","""Luz - MG""",17486,1762,1546,8841.358801
"""520320""","""Barro Alto - GO""",8716,795,766,8788.435062
…,…,…,…,…,…
"""521350""","""Monte Alegre de Goiás - GO""",7730,91,0,0.0
"""521385""","""Morro Agudo de Goiás - GO""",2356,0,0,0.0
"""521490""","""Nova Roma - GO""",3471,2,0,0.0


UF_codigo,notificacoes,casos_confirmados,populacao,taxa_confirmados_100k
str,u64,u64,u64,f64
"""17""",17299,8746,1383445,632.189932
"""32""",42085,21450,3514952,610.250154
"""15""",21872,12705,7581051,167.5889
"""51""",45740,35937,3035122,1184.038072
"""26""",59587,29376,8796448,333.952977
…,…,…,…,…
"""21""",6857,4810,6574789,73.158241
"""13""",9837,6392,3483985,183.468069
"""11""",28303,18465,1562409,1181.82883


## Distribuição espacial e rankings

Contagens acumuladas usam os anos-fonte presentes; taxas usam somente o ano selecionado e sua população. Os pontos são centroides da malha IBGE de 2022, apenas como localização de referência, sem harmonização de territórios históricos. Municípios sem correspondência são exibidos na auditoria e não entram nos mapas. Os rankings não dependem da cobertura da malha.

In [12]:
import geopandas as gpd
from utils.spatial_analysis import ensure_boundary_cache, _municipality_code_column, _normalize_municipality_codes

cumulative = panel.group_by("codigo_municipio").agg(pl.col("notificacoes").sum())
names = panel.filter(pl.col("municipio").is_not_null()).sort("ano").unique("codigo_municipio", keep="last").select("codigo_municipio", "municipio")
cumulative = cumulative.join(names, on="codigo_municipio", how="left", validate="1:1").with_columns(pl.col("municipio").fill_null(pl.col("codigo_municipio")))
period = f"{panel['ano'].min()}–{panel['ano'].max()} (anos-fonte disponíveis)"
boundaries = gpd.read_file(ensure_boundary_cache())
boundaries["codigo_municipio"] = _normalize_municipality_codes(boundaries[_municipality_code_column(boundaries)])
centroids = boundaries.to_crs(5880).geometry.centroid.to_crs(4326)
locations = pl.DataFrame({"codigo_municipio": boundaries["codigo_municipio"].tolist(), "latitude": centroids.y.tolist(), "longitude": centroids.x.tolist()})
display(cumulative.join(locations.select("codigo_municipio"), on="codigo_municipio", how="anti"))
display(municipal.join(locations.select("codigo_municipio"), on="codigo_municipio", how="anti"))
df_municipio = cumulative.join(locations, on="codigo_municipio", how="inner", validate="1:1").rename({"notificacoes": "Notificações", "municipio": "name"}).to_pandas()
df_municipio_relativo = municipal.join(locations, on="codigo_municipio", how="inner", validate="1:1").rename({"notificacoes": "Notificações", "populacao": "POPULACAO", "municipio": "name", "taxa_confirmados_100k": "taxa_100k"}).to_pandas()

codigo_municipio,notificacoes,municipio
str,u64,str
"""539909""",28,"""539909"""
"""520190""",1,"""520190"""
"""539907""",307,"""539907"""
"""521050""",1,"""521050"""
"""530180""",1,"""530180"""
…,…,…
"""140000""",10,"""140000"""
"""210000""",25,"""210000"""
"""410000""",12,"""410000"""


ano,codigo_municipio,municipio,populacao,fonte_populacao,notificacoes,casos_confirmados,notificacoes_data_invalida,notificacoes_codigo_notificacao_invalido,notificacoes_municipio_notificacao_divergente,municipios_notificacao_distintos,periodo_referencia,tem_notificacao_registrada,situacao_registro_sinan,correspondencia_populacao,taxa_notificacoes_100k,taxa_confirmados_100k
i32,str,str,u64,str,u64,u64,u64,u64,u64,u64,str,bool,str,bool,f64,f64


In [13]:
UF_CODES = {"11":"RO", "12":"AC", "13":"AM", "14":"RR", "15":"PA", "16":"AP", "17":"TO", "21":"MA", "22":"PI", "23":"CE", "24":"RN", "25":"PB", "26":"PE", "27":"AL", "28":"SE", "29":"BA", "31":"MG", "32":"ES", "33":"RJ", "35":"SP", "41":"PR", "42":"SC", "43":"RS", "50":"MS", "51":"MT", "52":"GO", "53":"DF"}
df_estado = cumulative.with_columns(pl.col("codigo_municipio").str.slice(0, 2).replace_strict(UF_CODES, default=None).alias("Estado")).group_by("Estado").agg(pl.col("notificacoes").sum().alias("Notificações")).to_pandas()
display(df_estado[df_estado["Estado"].isna()])
df_estado = df_estado.dropna(subset=["Estado"])
df_estado["Percentual"] = df_estado["Notificações"] / df_estado["Notificações"].sum() * 100
df_estado = df_estado.sort_values("Notificações", ascending=False)
fig = px.bar(df_estado, x="Estado", y="Notificações", color="Notificações", color_continuous_scale="Reds",
             text=df_estado["Percentual"].map(lambda x: f"{x:.1f}%"),
             title=f"Distribuição de notificações por UF de residência — {period}")
fig.update_traces(textposition="outside")
fig.update_layout(xaxis_title="Estado (UF)", yaxis_title="Número de notificações", showlegend=False)
fig.show()

,Estado,Notificações
0,None,1789


In [14]:
fig = px.scatter_map(
    df_municipio,
    lat='latitude',
    lon='longitude',
    size='Notificações',
    hover_name='name',
    size_max=30,
    zoom=3,
    map_style="carto-positron",
    title=f"Notificações acumuladas por município — {period}",
    height=800
)

fig.update_traces(marker=dict(color='red'))

fig.show()

In [15]:
fig = px.scatter_map(
    df_municipio_relativo,
    lat='latitude',
    lon='longitude',
    size='taxa_100k',
    hover_name='name',
    hover_data={
        'Notificações': True,
        'POPULACAO': True,
        'taxa_100k': ':.2f'
    },
    size_max=30,
    zoom=3,
    map_style="carto-positron",
    title=f"Confirmados por 100 mil residentes — ano-fonte {YEAR}"
)

fig.update_traces(marker=dict(color='red'))

fig.update_layout(
    width=1200,
    height=800
)

fig.show()

In [16]:
top_mun = cumulative.rename({'notificacoes':'Notificações', 'municipio':'name'}).to_pandas().sort_values(
    'Notificações',
    ascending=False
).head(10)

top_mun['Notificações'] = top_mun['Notificações'].astype(float)

fig = px.bar(
    top_mun.sort_values('Notificações'),
    x='Notificações',
    y='name',
    orientation='h',
    text='Notificações',
    title=f"Top 10 municípios por notificações — {period}",
    labels={
        'Notificações': 'Número de notificações',
        'name': 'Município'
    },
    color='Notificações'
)

fig.update_traces(textposition='outside')

fig.show()

In [17]:
top_mun_relativo = municipal.rename({'taxa_confirmados_100k':'taxa_100k', 'municipio':'name'}).to_pandas().sort_values(
    'taxa_100k',
    ascending=False
).head(10)

fig = px.bar(
    top_mun_relativo.sort_values('taxa_100k'),
    x='taxa_100k',
    y='name',
    orientation='h',
    text=top_mun_relativo['taxa_100k'].round(2),
    title=f"Top 10 por confirmados por 100 mil residentes — ano-fonte {YEAR}",
    labels={
        'taxa_100k': 'Confirmados por 100 mil residentes',
        'name': 'Município'
    },
    color='taxa_100k'
)

fig.update_traces(textposition='outside')

fig.show()

## Conclusões

A série corrigida usa classificação final em todas as eras. As contagens antigas baseadas no antecedente `DENGUE` não fundamentam conclusões sobre magnitude ou tendência em 2000–2006. O período anual é o ano-fonte, e a localização é a residência. As taxas são descritivas: refletem vigilância, diagnóstico e transmissão, sem demonstrar causalidade ou eliminar subnotificação.